Click "Copy to Drive" above to copy this file to your Colab account.


##Stepwise selection - brute force approach

In [ ]:
# -*- coding: utf-8 -*-


import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)
import scipy.stats as stats
import seaborn as sns

import statsmodels.api as sm



# for regression
import statsmodels.formula.api as smf
from sklearn import datasets
from statistics import mean



Read Data, DU

In [ ]:
#Read the data, modified USCrime data

data=pd.read_csv("https://raw.githubusercontent.com/DataAnalytics808/DASC-522-demo-repository/main/data/UScrime%202.csv")

print(data.shape)
print(list(data.columns))

X = data.loc[:, data.columns != 'Crime']
y = data.loc[:, data.columns == 'Crime']
X=sm.add_constant(X)

y

DU2

In [ ]:
X

Create LR model

In [ ]:
model = smf.ols('y ~ M + So + Ed + Po1 + LF + MF + Pop + NW + U1 + Wealth + Ineq + Prob + Time', data=data).fit()

# model = smf.ols('y ~ M + Ed + Po1 + Wealth + Ineq + Prob ', data=data).fit()

predictions = model.predict(X)

print_model = model.summary()
print(print_model)


Model 1

In [ ]:
###  Build Every Possible Model
from itertools import chain, combinations

def powerset(iterable):
    s=list(iterable)
    return chain.from_iterable(combinations(s,r) for r in range(1, len(s)+1))

mylist=list(powerset(list(X.columns)))
mylist=[list(row) for row in mylist]

print(mylist)

Analysis - AIC & BIC

In [ ]:
##Target is AIC
AIC_scores=pd.DataFrame(columns=["AIC"])
for i in range(len(mylist)):
    AIC_scores.loc[i,'AIC']=sm.OLS(y,X[mylist[i]]).fit().aic

print(AIC_scores.sort_values(by='AIC').head(50))



#             AIC
# 6885   641.154
# 6851   641.208
# 10456  641.235
# 3670   641.464
# 10476  641.626

mylist[6885]


# Out[6]: ['const', 'M', 'Ed', 'Po1', 'Wealth', 'Ineq', 'Prob']
# mylist[6851]
# Out[7]: ['const', 'M', 'Ed', 'Po1', 'M.F', 'Ineq', 'Prob']
# mylist[10456]
# Out[8]: ['const', 'M', 'Ed', 'Po1', 'M.F', 'Wealth', 'Ineq', 'Prob']
# mylist[3670]
# Out[9]: ['const', 'M', 'Ed', 'Po1', 'Ineq', 'Prob']
# mylist[10476]
# Out[10]: ['const', 'M', 'Ed', 'Po1', 'Pop', 'Wealth', 'Ineq', 'Prob']

BIC analysis

In [ ]:
##Target is BIC
BIC_scores=pd.DataFrame(columns=["BIC"])
for i in range(len(mylist)):
    BIC_scores.loc[i,'BIC']=sm.OLS(y,X[mylist[i]]).fit().bic

print(BIC_scores.sort_values(by='BIC').head(50))

mylist[3670]
#            BIC
# 3670   652.565
# 6885   654.105
# 6851   654.159
# 3667    654.97
# 1531   655.001
# 6882   655.031
# 1888   655.102

In [ ]:
len(mylist)
mylist[3670]

Stepwise Selection - Select K-best - only for classification tasks

In [ ]:
#### Select K Best
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_regression

# feature extraction
test = SelectKBest(score_func=f_regression, k=7)
fit = test.fit(X, y)

# summarize scores
np.set_printoptions(precision=3)
print(fit.scores_)
features = fit.transform(X)


# summarize selected features
print(features[0:5,:])

#Selected Ed, Po1, LF, MF, NW, Wealth, prob



Recursive Feature Elimination

In [ ]:
###  Recursive Feature Elimination

from sklearn.feature_selection import RFE
from sklearn import linear_model
model = linear_model.LinearRegression()

numberOfFeaturesToSelect = 5

# rfe = RFE(model, numberOfFeaturesToSelect)
rfe = RFE(estimator = model, n_features_to_select = numberOfFeaturesToSelect)
fit = rfe.fit(X, y)

f = fit.get_support(1) #the most important features
# final_features = data[data.columns[f]] # final features: this gives wrong results
final_features = X[X.columns[f]] # final features

print("Num Features: %d" % fit.n_features_)
print("Selected Features: %s" % final_features.columns)
print("Score: %2.3f" % fit.score(X,y))

print("----------")

#Selected 'Ed', 'LF', 'MF', 'Wealth', 'Time'

# summarize all features
for i in range(X.shape[1]):
	print('Column: %d, %s, Selected %s, Rank: %.3f' % (i, X.columns.values[i],rfe.support_[i], rfe.ranking_[i]))


Model the result

In [ ]:
# model = smf.ols('y ~ Prob', data=data).fit()  # best 1
# model = smf.ols('y ~ Prob + U1', data=data).fit()   # best 2
model = smf.ols('y ~ So + Po1 + LF + U1 + Prob', data=data).fit()   # best 5

predictions = model.predict(X)

print_model = model.summary()
print(print_model)

RFE iteration

In [ ]:
model = linear_model.LinearRegression()


print("\nSearch the best k features for k = 1 to 12\n")

for k in range(1, 12):
  # rfe = RFE(model, k)
  rfe = RFE(estimator = model, n_features_to_select = k)

  fit = rfe.fit(X, y)

  f = fit.get_support(1) #the most important features

  # final_features = data[data.columns[f]] # final features: this gives wrong results
  final_features = X[X.columns[f]] # final features

  print("Num Features: %d" % fit.n_features_)
  print("Selected Features: %s" % final_features.columns)
  print("Score: %2.2f" % fit.score(X,y))
  print("----------")
